# 09 - Adapting the workflow to a new sensor

This notebook is a template/checklist, not a runnable pipeline. It walks
through how to adapt the chlorophyll reconstruction workflow in this
repository to a different sensor variable -- for example, dissolved
oxygen, which is planned as a second case study at the same site but is
not yet built. Most of this notebook is markdown guidance referencing the
methodology docs; only the first cell loads anything.


## Step 0: Confirm this repository's scope

Read `docs/methodology/target_and_gap_construction.md`,
`docs/methodology/validation_protocol.md`, and `docs/evidence_hierarchy.md`
before adapting anything. The target definition, eligibility rule, and
validation protocol are the load-bearing scientific decisions in this
workflow; copying the code without understanding these choices risks
silently producing invalid results for a new sensor.


## Step 1: Define the new target

For a new sensor (e.g. oxygen), you will need to make the same decisions
documented in `docs/methodology/target_and_gap_construction.md` for
chlorophyll:

- What is the daily aggregation rule (mean? median?) and why?
- What hourly-validity threshold defines an "eligible" day?
- What is the full valid range, and how are invalid/negative/out-of-range
  raw readings handled?
- Are there known sensor drift, calibration, or fouling issues specific to
  this variable that need a different QA approach than chlorophyll?

These are scientific decisions, not implementation details -- they should
be made deliberately and documented, not inherited by default from the
chlorophyll target's design choices.


In [ ]:
# Once the new daily target table exists in the same schema as
# chlorophyll_daily_target.csv (date, <new_target>_mean, target_eligible_default, ...),
# it can be loaded with the same utility:

import sys
sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_daily_target

# new_target_df = load_daily_target("path/to/new_sensor_daily_target.csv")


## Step 2: Re-run the gap audit

Use `notebooks/01_target_and_gap_audit.ipynb` as a template: regenerate
coverage statistics, real-gap inventory, and eligible-run structure for the
new target. Missingness patterns may differ substantially between sensors
(e.g. oxygen sensors may have different fouling/drift failure modes than
chlorophyll fluorometers).


## Step 3: Reconstruct the artificial-gap validation pool

Use `src/coastal_gap_reconstruction/artificial_gap_validation.py`'s
`generate_gap_candidates` function against the new target's eligible-run
structure. Decide whether the same gap lengths (1, 3, 7, 14, 30, 45, 60
days) remain appropriate, or whether the new sensor's missingness pattern
calls for a different set.


## Step 4: Re-evaluate baselines first

Run `notebooks/03_baselines.ipynb`'s logic against the new target before
trying anything more sophisticated. The relative ranking of climatology
vs. persistence vs. interpolation may differ for a variable with different
seasonal/autocorrelation structure than chlorophyll.


## Step 5: Re-select predictor features

The curated chlorophyll feature table includes chlorophyll-specific
covariates (a satellite chlorophyll proxy, upwelling indices tuned for
biological productivity). For a new sensor, re-evaluate which external
predictors are physically relevant -- do not assume the same feature table
transfers without justification.


## Step 6: Re-run engineered tabular / TS-ICL methods

Once a baseline floor and a relevant feature set exist for the new sensor,
revisit `docs/methodology/model_families.md` and
`notebooks/06_tsicl_zero_shot_imputation.ipynb` to apply the same model
ladder. Re-check whether a satellite proxy covariate is meaningful for the
new variable -- for oxygen, for example, there is no obvious "satellite
oxygen proxy" analogous to satellite chlorophyll, so the leading TS-ICL
configuration for chlorophyll may not transfer directly.


## Step 7: Re-establish the evidence hierarchy

Apply the same discipline described in `docs/evidence_hierarchy.md`:
artificial-gap validation results are the only validation-grade evidence;
real-gap candidate outputs are plausibility only. This discipline does not
change across sensors.
